# Week 9 lab · Scrape a table and store it efficiently

**CMT309/509 · CMT124** · one-hour lab

You will run the whole pipeline from this week on one real page: **download** (a saved snapshot), **extract** a Wikipedia table with BeautifulSoup, **store** it as CSV and Parquet, and **analyse** it with a chart.

**How this lab works**
* Work through the tasks in order. Each task has an empty cell for your code, followed by a **checkpoint** cell.
* Run the checkpoint after your code. If it prints `✔`, move on. If it raises an `AssertionError`, read the message, fix your code and try again.
* **Question** cells ask you to explain a result in a sentence or two. Write your answer in the text cell (double-click to edit).
* The **Challenge** at the end is optional and needs an internet connection.

**Data:** Wikipedia, *List of countries by population growth rate*, a saved snapshot of the page as it was on 21 February 2025, licensed under CC BY-SA 4.0. Because it is a snapshot, the row numbers in this lab always match.

## Setup

Run this cell first. It loads the snapshot as one long string of HTML, `html`.

In [ ]:
# --- Setup: run this cell first -------------------------------------------------
# Course datasets live in the course repository under 2026-27/datasets/.
# get_data() uses a local copy if the notebook runs inside a copy of the repository;
# otherwise (e.g. in Google Colab) it downloads the file once and reuses it.
import os
import urllib.request

COURSE_REPO = "OWNER/REPO"
BASE_URL = f"https://raw.githubusercontent.com/{COURSE_REPO}/main/2026-27/"


def course_file(relpath):
    """Return a local path to a course file such as 'datasets/owid_co2_cleaned.csv'."""
    for root in ("../../..", "."):                     # 1) running inside a copy of the repository
        path = os.path.join(root, relpath)
        if os.path.exists(path):
            return path
    path = os.path.join("course_files", relpath)       # 2) Colab: download once
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(BASE_URL + relpath, path)
    return path


def get_data(name):
    """Return a local path to a course dataset, e.g. get_data('owid_co2_cleaned.csv')."""
    return course_file("datasets/" + name)

import re
import time

import pandas as pd
import matplotlib.pyplot as plt
import requests
from bs4 import BeautifulSoup

html = open(get_data("wiki_population_growth_rate_2025-02-21.html"), encoding="utf-8").read()
print(len(html), "characters of HTML")

## Task 1 · Find the table and its rows

1. Parse `html` with BeautifulSoup and store the result in `soup`.
2. The page has more than one table. Find the **first table with the class `wikitable`** and store it in `table`.
3. Store all the rows (`tr`) **of that table** in `rows`, and print how many there are.

Then look at `rows[0]`, `rows[1]` and `rows[2]`: which of them are header rows, and what is in `rows[2]`?

In [ ]:
# STUDENT TASK
# Task: soup, table (class wikitable) and rows (the tr tags inside table)
# Hint: soup.find("table", class_="wikitable") and table.find_all("tr")
# your code here


In [ ]:
# Checkpoint 1
assert table is not None and table.name == "table", "table: use soup.find('table', class_='wikitable')"
assert len(rows) == 243, "rows: find the tr tags inside table only (2 header rows, World, 240 countries)"
print("✔ Task 1")

## Task 2 · Column names

The table has **two header rows**. `rows[0]` holds six header cells (`th`), but the *CIA* header spans two columns, and `rows[1]` names those two: `(%)` and `Year`.

1. Make a list `headers` with the text of every `th` in `rows[0]`, with the footnote markers such as `[2][3]` removed and spaces stripped. *Hint:* `re.sub(r"\[\d+\]", "", text)` from Week 8.
2. Build the final list `cols` of **seven** names: the first header, then `"CIA (%)"` and `"CIA year"` in place of the CIA header, then the other four headers unchanged.

In [ ]:
# STUDENT TASK
# Task: headers (cleaned th texts of rows[0]) and cols (7 names)
# Hint: list slicing and + join lists: [headers[0]] + [...] + headers[2:]
# your code here


In [ ]:
# Checkpoint 2
assert len(headers) == 6, "headers: one name per th in rows[0]"
assert not any("[" in h for h in headers), "headers: remove the footnote markers like [2][3]"
assert len(cols) == 7 and cols[0] == "Country (or territory)", "cols: seven names, starting with the country column"
assert cols[1:3] == ["CIA (%)", "CIA year"] and cols[3] == "WB 2023", "cols: CIA (%), CIA year, then the remaining headers"
print("✔ Task 2")

## Task 3 · One row first

Before you loop over 240 rows, get **one** row right. `rows[3]` is Afghanistan.

In each country row the **name is in a `th` cell** and the six values are in `td` cells.

1. Store the cleaned country name of `rows[3]` in `name`.
2. Store a list of the six cell texts (stripped) in `values`.
3. Now try `rows[4]` (Albania): its name comes out as `'Albania *'`. Change your name-cleaning so that everything from the `*` onwards is removed, and store the clean name in `name4`. *Hint:* `text.split("*")[0].strip()`.

In [ ]:
# STUDENT TASK
# Task: name, values (rows[3]) and name4 (rows[4], without the '*')
# Hint: row.find("th").text, [td.text.strip() for td in row.find_all("td")]
# your code here


In [ ]:
# Checkpoint 3
assert name == "Afghanistan", "name: the text of the th cell, stripped"
assert values == ["2.22", "2024", "2.7", "2.78", "3.16", "2.41"], "values: the six td texts of rows[3], stripped"
assert name4 == "Albania", "name4: remove the '*' and everything after it, then strip"
print("✔ Task 3")

## Task 4 · All rows → DataFrame

1. Loop over the **country rows** (`rows[3:]`; `rows[2]` is the World total). For each row, build a list `[name] + values` with the cleaning from Task 3, and append it to a list `records`.
2. Make a DataFrame `growth` from `records` with the column names `cols`.
3. Every column except the first holds numbers as **text**, and some cells are empty (`''`). Convert columns 1 to 6 to numbers with `pd.to_numeric(growth[col], errors="coerce")`, which turns empty cells into `NaN`.

In [ ]:
# STUDENT TASK
# Task: records -> growth (240 rows, 7 columns) -> numeric columns
# Hint: for col in cols[1:]: growth[col] = pd.to_numeric(growth[col], errors="coerce")
# your code here


In [ ]:
# Checkpoint 4
assert growth.shape == (240, 7), f"growth should have 240 rows and 7 columns, not {growth.shape}"
assert growth["CIA (%)"].dtype == "float64", "convert the value columns with pd.to_numeric(..., errors='coerce')"
assert growth.loc[growth[cols[0]] == "Afghanistan", "CIA (%)"].item() == 2.22, "Afghanistan's CIA (%) should be 2.22"
assert growth["CIA (%)"].isna().sum() == 10, "10 countries have no CIA value: empty cells should become NaN"
assert not growth[cols[0]].str.contains(r"\*").any(), "some names still contain '*'"
print("✔ Task 4")

In [ ]:
# If your result does not match the checkpoint, run this cell to continue.
rows = BeautifulSoup(html, "html.parser").find("table", class_="wikitable").find_all("tr")
cols = ["Country (or territory)", "CIA (%)", "CIA year", "WB 2023", "UN 2005–10", "UN 2010–15", "UN 2015–20"]
records = []
for row in rows[3:]:
    records.append([row.find("th").text.split("*")[0].strip()] + [td.text.strip() for td in row.find_all("td")])
growth = pd.DataFrame(records, columns=cols)
for col in cols[1:]:
    growth[col] = pd.to_numeric(growth[col], errors="coerce")

**Question 4.** Why does `growth["CIA (%)"].astype(float)` fail on the scraped text, when `pd.to_numeric(..., errors="coerce")` works? What does each `NaN` in `growth` mean?

*Your answer:*

## Task 5 · Store it: CSV and Parquet

1. Save `growth` as `growth.csv` and as `growth.parquet` (both with `index=False`).
2. Print the size of each file in bytes (`os.path.getsize`).
3. Time how long each file takes to read back, with `time.perf_counter()` before and after `pd.read_csv` / `pd.read_parquet` (or `%time`).
4. Reload the table from the **Parquet** file into `reloaded`.

In [ ]:
# STUDENT TASK
# Task: save CSV + Parquet, compare sizes and read times, reload from Parquet
# Hint: start = time.perf_counter(); ...; print(time.perf_counter() - start)
# your code here


In [ ]:
# Checkpoint 5
assert os.path.exists("growth.csv") and os.path.exists("growth.parquet"), "save both files"
assert reloaded.shape == (240, 7), "reloaded: read growth.parquet back"
assert list(reloaded.columns) == cols, "reloaded: the column names should survive the round trip (did you save with index=False?)"
assert reloaded["CIA (%)"].dtype == "float64", "Parquet keeps the number type"
print("✔ Task 5")

**Question 5.** In the lecture, Parquet was about six times smaller than CSV and faster to read. Which file is smaller **here**, and is Parquet faster? Explain the difference.

*Your answer:*

## Task 6 · The ten fastest-growing countries

1. Store in `top10` the **ten countries with the highest `CIA (%)`**, highest first (use `reloaded`).
2. Draw them as a horizontal bar chart with matplotlib (`fig, ax = plt.subplots()`, `ax.barh`), largest at the top, with an x-axis label and a title.

In [ ]:
# STUDENT TASK
# Task: top10 (sorted, highest first) and a horizontal bar chart
# Hint: sort_values("CIA (%)", ascending=False).head(10); ax.invert_yaxis() puts the first bar on top
# your code here


In [ ]:
# Checkpoint 6
assert len(top10) == 10, "top10: ten rows"
assert top10.iloc[0][cols[0]] == "Syria" and top10.iloc[0]["CIA (%)"] == 6.39, "top10: sort by CIA (%), highest first"
assert top10["CIA (%)"].is_monotonic_decreasing, "top10: highest first"
print("✔ Task 6")

**Question 6.** Look at the top three countries in all the value columns: `top10.head(3)`. Do the sources (CIA, World Bank, UN) agree? Give one reason why estimates of the same quantity can differ so much.

*Your answer:*

## Challenge (optional) · The live page

The snapshot is from February 2025. Request **today's** page and compare it with the snapshot.

1. Request `https://en.wikipedia.org/wiki/List_of_countries_by_population_growth_rate` with `requests.get(url, headers=..., timeout=15)`. Send an **honest User-Agent** that names your program and how to contact you, e.g. `"W09-lab/1.0 (Cardiff University student exercise; you@example.ac.uk)"`. (Without one, Wikipedia answers with **HTTP 403 Forbidden**. Try it and look at `response.status_code`.)
2. Wrap the request in `try`/`except` so the notebook still runs without internet.
3. Parse the live page, find the first `table`, and compare its header texts and its number of rows with the snapshot. Would your Task 2–4 code still work on today's page?

Request the page **once**: you don't need to download it again to explore it.

In [ ]:
# STUDENT TASK
# Task: request the live page politely and compare its first table with the snapshot
# Hint: response.raise_for_status() turns a 403 or 404 into an exception
# your code here


## Wrap-up

You found a table inside a full web page, rebuilt its two-row header, scraped 240 irregular rows into a typed DataFrame, stored it as CSV and Parquet (and saw that Parquet pays off only at scale), and turned it into a chart. That is the whole pipeline: ingestion → storage → processing → analysis → visualisation.